In [1]:
import mlflow
mlflow.set_tracking_uri('http://3.111.55.134:5000')

c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning.")

<Experiment: artifact_location='s3://mlflow-buckets-areeba/10', creation_time=1784308564725, effective_trace_archival_retention=None, experiment_id='10', last_update_time=1784308564725, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning.', tags={}, trace_location=None, workspace='default'>

In [4]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna

In [6]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Logistic Regression
# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")

# Step 6: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    # 'saga' supports l1/l2 penalties AND multiclass classification (liblinear does not)
    model = LogisticRegression(C=C, penalty=penalty, solver='saga', max_iter=2000, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# Step 7: Run Optuna for Logistic Regression, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=30)

    best_params = study.best_params
    best_model = LogisticRegression(
        C=best_params['C'],
        penalty=best_params['penalty'],
        solver='saga',
        max_iter=2000,
        random_state=42
    )

    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Logistic Regression
run_optuna_experiment()

[I 2026-07-17 23:39:43,721] A new study created in memory with name: no-name-f9e00fcc-aa08-4fe3-9afd-ff21b37598bf
c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
[I 2026-07-17 23:39:44,843] Trial 0 finished with value: 0.6688860705981822 and parameters: {'C': 0.0054719462572988984, 'penalty': 'l2'}. Best is trial 0 with value: 0.6688860705981822.
c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecate

🏃 View run LogisticRegression_SMOTE_TFIDF_Trigrams at: http://3.111.55.134:5000/#/experiments/10/runs/27610ba48da340939f2d5af26b1160f1
🧪 View experiment at: http://3.111.55.134:5000/#/experiments/10
